In [ ]:
import sys, os; sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__) if '__file__' in globals() else os.getcwd(), '..')))
#import os; os.chdir(os.path.dirname(os.getcwd()))
from utils.model_loader import get_model_fits
import numpy as np
import pandas as pd
import re
from sklearn.metrics import mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
results_dir_tanh = "results/classification/single_layer/tanh/breastcancer"

model_names_tanh = ["Gaussian", "RHS", "DHS", "DST"]

full_config_path = "breast_cancer_N455_p30"

tanh_fits = get_model_fits(
    config=full_config_path,
    results_dir=results_dir_tanh,
    models=model_names_tanh,
    include_prior=False,
)


In [ ]:
from utils.generate_data import load_breast_cancer_data
X_train, X_test, y_train, y_test, *_ = load_breast_cancer_data(
    test_size=0.2, standardize=False, random_state=42
)

In [ ]:
import numpy as np

def expected_calibration_error(probs, y_true, n_bins=15, strategy="uniform"):
    """
    Compute ECE (and MCE) using the standard 'confidence' binning approach.

    Parameters
    ----------
    probs : array, shape (n_samples, n_classes) or (n_samples,)
        Predicted probabilities per class (multiclass) or positive-class probs (binary).
    y_true : array, shape (n_samples,)
        True labels as integers in [0, n_classes-1].
    n_bins : int
        Number of bins in [0, 1].
    strategy : {'uniform', 'quantile'}
        'uniform' uses equal-width bins; 'quantile' uses equal-mass bins based on confidences.

    Returns
    -------
    ece : float
        Expected Calibration Error (weighted average |acc - conf|).
    mce : float
        Maximum Calibration Error (max |acc - conf| across bins).
    bin_stats : dict
        Per-bin counts, accuracy, confidence, and edges.
    """
    probs = np.asarray(probs)
    y_true = np.asarray(y_true)

    # Convert to confidences (max class prob) and predicted labels
    if probs.ndim == 1 or (probs.ndim == 2 and probs.shape[1] == 1):
        # Binary: probs is P(y=1). Turn into confidences wrt predicted label.
        p1 = probs.ravel()
        y_hat = (p1 >= 0.5).astype(int)
        conf = np.where(y_hat == 1, p1, 1.0 - p1)
    else:
        y_hat = probs.argmax(axis=1)
        conf = probs.max(axis=1)

    n = len(y_true)
    if strategy == "uniform":
        edges = np.linspace(0.0, 1.0, n_bins + 1)
    elif strategy == "quantile":
        # Use unique quantiles to avoid duplicate edges when many equal confidences
        quantiles = np.linspace(0.0, 1.0, n_bins + 1)
        edges = np.unique(np.quantile(conf, quantiles))
        # Ensure we still cover [0,1]
        edges[0], edges[-1] = 0.0, 1.0
    else:
        raise ValueError("strategy must be 'uniform' or 'quantile'")

    ece = 0.0
    mce = 0.0
    bin_counts, bin_accs, bin_confs = [], [], []

    for b in range(len(edges) - 1):
        lo, hi = edges[b], edges[b + 1]
        # Include left, exclude right except for final bin
        if b < len(edges) - 2:
            mask = (conf >= lo) & (conf < hi)
        else:
            mask = (conf >= lo) & (conf <= hi)

        count = int(mask.sum())
        if count == 0:
            bin_counts.append(0)
            bin_accs.append(np.nan)
            bin_confs.append(np.nan)
            continue

        acc_b = (y_hat[mask] == y_true[mask]).mean()
        conf_b = conf[mask].mean()
        gap = abs(acc_b - conf_b)

        weight = count / n
        ece += weight * gap
        mce = max(mce, gap)

        bin_counts.append(count)
        bin_accs.append(acc_b)
        bin_confs.append(conf_b)

    bin_stats = {
        "counts": np.array(bin_counts),
        "acc": np.array(bin_accs),
        "conf": np.array(bin_confs),
        "edges": np.array(edges),
    }
    return float(ece), float(mce), bin_stats


In [ ]:
from sklearn.metrics import accuracy_score, log_loss
from scipy.stats import mode
import pandas as pd
import numpy as np

model_names = list(tanh_fits.keys())
results = []

for model in model_names:
    posterior = tanh_fits[model]['posterior']
    
    # Predicted class labels from posterior samples: majority vote
    pred_samples = posterior.stan_variable("pred_test")                 # shape: [n_samples, n_test]
    majority_preds = mode(pred_samples, axis=0, keepdims=False).mode.flatten()

    # Accuracy
    acc = accuracy_score(y_test, majority_preds)

    # Mean predictive probabilities across posterior samples
    pred_probs = posterior.stan_variable("prob_test")                   # shape: [n_samples, n_test, n_classes]
    mean_probs = pred_probs.mean(axis=0)                                 # shape: [n_test, n_classes]

    # Negative log-likelihood (log loss)
    y_test_adj = y_test - 1  # ensure labels are {0,1} for binary
    nll = log_loss(y_test_adj, mean_probs, labels=[0, 1])

    # ECE / MCE (you can change n_bins or strategy)
    ece, mce, _ = expected_calibration_error(mean_probs, y_test_adj, n_bins=15, strategy="uniform")

    results.append({
        "Model": model,
        "Accuracy": acc,
        "NLL": nll,
        "ECE": ece,
        "MCE": mce
    })

results_df = pd.DataFrame(results)


In [ ]:
latex_table = results_df.to_latex(index=False, float_format="%.4f", column_format="lcc", caption="Accuracy and NLL per model.", label="tab:accuracy_nll")
print(latex_table)

In [ ]:
print("Of 114 observations,", np.round(114*results_df['Accuracy'][0], 3), "were classified correctly by the", results_df['Model'][0], "model \n")
print("Of 114 observations,", np.round(114*results_df['Accuracy'][1], 3), "were classified correctly by the", results_df['Model'][1], "model \n")
print("Of 114 observations,", np.round(114*results_df['Accuracy'][2], 3), "were classified correctly by the", results_df['Model'][2], "model \n")
print("Of 114 observations,", np.round(114*results_df['Accuracy'][3], 3), "were classified correctly by the", results_df['Model'][3], "model \n")
# print("Of 114 observations,", np.round(114*results_df['Accuracy'][4], 3), "were classified correctly by the", results_df['Model'][4], "model \n")
# print("Of 114 observations,", np.round(114*results_df['Accuracy'][5], 3), "were classified correctly by the", results_df['Model'][5], "model \n")
#print("Of 114 observations,", np.round(114*results_df['Accuracy'][3], 3), "were classified correctly by the", results_df['Model'][4], "model \n")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

X_test_df = pd.DataFrame(X_test)

W_1 = tanh_fits['Gaussian']['posterior'].stan_variable("W_1")[0, :, :]   # (30, 16)
W_2 = tanh_fits['Gaussian']['posterior'].stan_variable("W_L")[0, :, :]   # (16, 2)
b_1 = tanh_fits['Gaussian']['posterior'].stan_variable("hidden_bias")[0, :]  # (16,)
b_2 = tanh_fits['Gaussian']['posterior'].stan_variable("output_bias")[0, :]  # (2,)

# Konverter DataFrame til tensor
X = torch.tensor(X_test_df.to_numpy(), dtype=torch.float32)  # (114, 30)

# Stan-vekter til tensor
W1 = torch.tensor(W_1, dtype=torch.float32)   # (30, 16)
b1 = torch.tensor(b_1.squeeze(), dtype=torch.float32)  # (16,)
W2 = torch.tensor(W_2, dtype=torch.float32)   # (16, 2)
b2 = torch.tensor(b_2, dtype=torch.float32)   # (2,)

# Definer nettverket
class StanNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, activation=torch.tanh):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, output_dim)
        self.activation = activation

    def forward(self, x):
        h = self.activation(self.linear1(x))
        logits = self.linear2(h)
        return logits  # matcher Stan sin `output`

    def predict_proba(self, x):
        logits = self.forward(x)
        return F.softmax(logits, dim=1)  # matcher Stan sin `prob_test`

    def predict(self, x):
        probs = self.predict_proba(x)
        return torch.argmax(probs, dim=1)  # matcher Stan sin `pred_test`

# Bygg og kopier vektene
model = StanNNClassifier(input_dim=30, hidden_dim=16, output_dim=2, activation=torch.tanh)
with torch.no_grad():
    model.linear1.weight.copy_(W1.T)   # (16, 30)
    model.linear1.bias.copy_(b1)       # (16,)
    model.linear2.weight.copy_(W2.T)   # (2, 16)
    model.linear2.bias.copy_(b2)       # (2,)

# === Test forward ===
logits = model(X)                # (114, 2), Stan: `output_test`
probs = model.predict_proba(X)   # (114, 2), Stan: `prob_test`
preds = model.predict(X)         # (114,),   Stan: `pred_test`

#print("logits shape:", logits.shape)
#print("probs shape :", probs)
#print("preds shape :", preds)


In [ ]:
stan_probs = tanh_fits['Gaussian']['posterior'].stan_variable("prob_test")[0, :, :]
stan_probs_torch = torch.tensor(stan_probs, dtype=torch.float32)

print("Max diff:", (probs - stan_probs_torch).abs().max().item())


In [ ]:
from collections import defaultdict
import pandas as pd
import torch.nn.functional as F
from utils.robust_utils import estimate_robustness_over_test_set
import torch
# === Settings ===
epsilons = [0.01, 0.05, 0.1, 0.25]
#epsilons = [0.1, 0.5, 1.0, 2.5]
scales = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
sample_indices = range(0, 4000, 800)
input_dim = X_test.shape[1]
hidden_dim = 16
output_dim = 2
p_norm = 2

# === Subset test set ===
X_test_df = pd.DataFrame(X_test)
y_test_s = pd.Series(y_test) - 1  # Ensure labels in {0,1}
test_subset = X_test_df.sample(frac=1.0, random_state=42)
subset_indices = test_subset.index
y_subset = y_test_s.loc[subset_indices]

# === Run robustness for each model ===
all_results = []

for model_name, fit_entry in tanh_fits.items():
    for epsilon in epsilons:
        for scale in scales:
            delta = scale * epsilon

            df_result = estimate_robustness_over_test_set(
                x_test=test_subset,
                y_test=y_subset,
                fits_dict={model_name: fit_entry},  # Wrap as dict
                input_dim=input_dim,
                hidden_dim=hidden_dim,
                output_dim=output_dim,
                sample_indices=sample_indices,
                epsilon=epsilon,
                delta=delta,
                p_norm=p_norm,
                activation=torch.tanh
            )

            df_result["epsilon"] = epsilon
            df_result["delta"] = delta
            df_result["scale"] = round(scale, 2)
            df_result["model"] = model_name
            all_results.append(df_result.copy())

# === Combine results ===
df_robust = pd.concat(all_results, ignore_index=True)

# === Unpack 'robustness' dictionary column into separate columns ===
df_flat = pd.concat(
    [df_robust.drop(columns=["robustness"]),
     df_robust["robustness"].apply(pd.Series)],
    axis=1
)

# === Add derived columns ===
df_flat["1-p1"] = (1.0 - df_flat["p1"]).round(5)
df_flat["1-p2"] = (1.0 - df_flat["p2"]).round(5)

# Resulting DataFrame: df_flat


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure model name consistency and ordering
# model_order = ["DHS", "DST", "Beta Horseshoe tanh", "Beta Student T tanh"]
# model_order = ["Dirichlet Horseshoe", "Dirichlet Student T", "Beta Horseshoe", "Beta Student T"]
model_order = ["Gaussian", "RHS", "DHS", "DST"]
# model_order = ["Gaussian", "Regularized Horseshoe", "Dirichlet Horseshoe", "Dirichlet Student T", "Beta Horseshoe", "Beta Student T"]

#Ns = sorted(df_flat["N"].unique())

# Global color scale (consistent across all plots)
vmin = df_flat["p1"].min()
vmax = df_flat["p1"].max()

# short_names = {
#     "Gaussian": "Gauss",
#     "Regularized Horseshoe": "RHS",
#     "Dirichlet Horseshoe": "DHS",
#     "Dirichlet Student T": "DST",
#     "Beta Horseshoe": "BHS",
#     "Beta Student T": "BST",
# }

short_names = {
    "Gaussian": "Gauss",
    "RHS": "RHS",
    "DHS": "DHS",
    "DST": "DST",
    "Beta Horseshoe tanh": "BHS",
    "Beta Student T tanh": "BST",
}

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 12), sharex=True, sharey=True)

for j, model in enumerate(model_order):
    row, col = divmod(j, 2)
    ax = axes[row, col]
    df_model = df_flat[df_flat["model"] == model]

    # Use pivot_table to handle duplicates
    heatmap_df = df_model.pivot_table(
        index="scale", columns="epsilon", values="p1", aggfunc="mean"
    )

    sns.heatmap(
        heatmap_df.sort_index(ascending=False),
        annot=True, fmt=".2f", cmap="RdYlGn_r", ax=ax,
        cbar=False, vmin=vmin, vmax=vmax, annot_kws={"fontsize":20}
    )

    #ax.set_title(model, fontsize=13)
    ax.set_title(short_names[model], fontsize=20)
    ax.set_xlabel(r"$\varepsilon$", fontsize=25)
    ax.set_ylabel(r"$\delta / \varepsilon$", fontsize=25)
    ax.tick_params(axis='x', labelsize=20)
    ax.tick_params(axis='y', labelsize=20)
    ax.tick_params(axis='both', which='major', labelsize=20)

# Adjust layout to leave space for colorbar
plt.tight_layout(rect=[0, 0, 0.93, 0.95])

# Add colorbar on the right
cbar_ax = fig.add_axes([0.94, 0.25, 0.015, 0.5])  # [left, bottom, width, height]
norm = plt.Normalize(vmin=vmin, vmax=vmax)
sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=norm)
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax, label='$p_1$')
fig.suptitle(f"Softmax Shift probability $p_1$", fontsize=20)
plt.savefig("figures_for_use_in_paper/breast_cancer_p1_tanh.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd

# assuming df is your big dataframe
summary = (
    df_flat.groupby("model")["p2"]
      .agg(["mean", "std", "min", "max"])
      .reset_index()
      .round(3)
)

print(summary)


In [ ]:
import numpy as np
import pandas as pd

def ecdf(values):
    x = np.sort(values)
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y


In [ ]:
import matplotlib.pyplot as plt

#df_plot = df_flat[df_flat["model"].str.contains("relu")]

plt.figure(figsize=(6, 5))

for model, g in df_flat.groupby("model"):
    x, y = ecdf(g["p2"].values)
    plt.step(x, y, where="post", label=model)

plt.xlabel(r"$p_2$")
plt.ylabel(r"$\mathbb{P}(p_2 < t)$")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
def safety_category_p2(p2):
    if p2 == 0.0:
        return "Safe"
    elif p2 == 1.0:
        return "Unsafe"
    else:
        return "Partially safe"

df_flat["safety"] = df_flat["p2"].apply(safety_category_p2)

summary = (
    df_flat
    .groupby(["model", "safety"])
    .size()
    .reset_index(name="count")
)

summary["fraction"] = summary.groupby("model")["count"].transform(
    lambda x: x / x.sum()
)

summary.sort_values(by="safety").round(3)


In [ ]:
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.barplot(
    data=summary,
    x="model",
    y="fraction",
    hue="safety"
)
plt.ylabel("Fraction of runs")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
